# TFC-WDA Baseline (Tunio et al., 2024) - Reproduction with REAL INT8 TFLite quantization

**Purpose.** This notebook re-implements the method of Tunio et al. (2024), *Computers and Electronics in
Agriculture* 227:109574, "Advancing plant disease classification: A robust and generalized approach with
transformer-fused convolution and Wasserstein domain adaptation". It runs on your exact PlantVillage -> PlantDoc
3-class split, so it can be compared with MicroBrainNet-DA (im-main) under an identical-data protocol.

**What changed compared with the previous version of this notebook (the point of this file).**
The previous version never quantized anything: "INT8 size" was `params x 1 byte`. This version produces a **measured,
full-integer INT8 TFLite model for every seed** with the **same export route as im-main**, and evaluates it:

| step | im-main | this notebook |
|---|---|---|
| PyTorch -> TF | rebuild the architecture natively in TF/Keras and copy the weights by hand | same, generic module-by-module translator |
| ONNX / onnx2tf | **not used for quantization** (its ONNX file is only a side artifact) | **not used at all** |
| Quantizer | `tf.lite.TFLiteConverter`, `Optimize.DEFAULT`, `TFLITE_BUILTINS_INT8`, int8 input and output | identical |
| Calibration | representative set from **both domains**, validation splits only (test never touched) | same recipe: 400 source-val + all 79 target-val images = 479 (im-main: 511) |
| Preprocessing | normalization baked into the graph, model takes raw [0,1] pixels | identical (ImageNet mean/std baked in) |
| Reported size | measured `.tflite` file | measured `.tflite` file |
| Reported accuracy | accuracy of the real INT8 interpreter on the target test set | identical, 3 seeds, bootstrap CI, per-image predictions saved |

A smoke test (Part 6b) builds, converts and runs the whole export on an untrained network **before** the hours-long
training starts, so a converter problem shows up in minutes, not after training.

**What is reproduced (from the paper text).**
- MobileNetV2 CNN branch (ImageNet-pretrained, Table 2) and a MobileViT branch (L = 2, 3, 4 transformer layers, 2x2 patches, Fig. 4).
- Feature fusing between the two branches at three scales: 1x1 conv channel alignment, stride-2 conv for down-sampling,
  interpolation for odd grids. Local details go into the transformer branch and global context goes back into the CNN branch (Sec. 3.2.2).
- One FC classifier per branch. Prediction = mean of the two logit vectors.
- Wasserstein-distance-informed representation learning (WDIRL, Eqs. 8-12): a domain critic with a gradient penalty,
  trained several steps per feature-extractor step. The extractor minimizes the critic's Wasserstein estimate.
- Uncertainty Penalty Loss (UPL, Eq. 14): complement-entropy term on source predictions.
- Elastic-net regularization (Eqs. 5-7) as the "NFRegNet" ingredient.
- Class-balanced weighted random sampling of the source loader (Sec. 3.1).
- Gaussian + median filtering of PlantDoc images (Sec. 3.1).
- Table 2 settings: Adam, lr 3e-4, batch 32, 50 epochs, 224 px, horizontal flip + rotation.
- Non-adversarial-label use: target labels never enter the training loss.

**What the paper does not specify, or states inconsistently (assumed here, all are config constants).**
- Critic architecture, number of critic steps, gradient-penalty weight, critic learning rate.
- lambda (Wasserstein weight) and alpha (UPL weight). lambda uses the usual 0 -> max ramp.
- Elastic-net strengths, MobileViT widths, head count, dropout.
- Sign conventions in Eq. 14 and Eq. 15 (see caveats). CDAN is mentioned in the Introduction but is not part of Eq. 15, so it is not implemented.

No official code link is given in the paper. This is a reconstruction from the text.

**Protocol parity with your other baselines.**
- Same `LeafIndex` logic and same `stratified_split_indices(seed=42)`.
- Same 3 classes. Same split sizes: 2628/564/564 source and 364/79/79 target.
- Same "smoothed target-val accuracy" checkpoint rule (window 5, patience 15).
- Same bootstrap 95% CI. 3 seeds (42, 1337, 2024), the same as im-main and the EffNet baseline.

Run this in the same Kaggle environment as the im-main notebook (TensorFlow is preinstalled on the Kaggle image; nothing needs to be pip-installed,
in particular no onnx / onnx2tf).

In [1]:
# ============================================================
# PART 0: IMPORTS AND CONFIGURATION
# ============================================================
import os, time, random, warnings, copy, math, gc, traceback
from collections import Counter

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.models as tvm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             balanced_accuracy_score, cohen_kappa_score, confusion_matrix)

# TensorFlow is used ONLY for the TFLite export / INT8 evaluation (CPU). It is hidden from the GPU so
# that it cannot grab the GPU memory that PyTorch needs for training.
import tensorflow as tf
try:
    tf.config.set_visible_devices([], "GPU")
except Exception as _e:
    print("note: could not hide GPUs from TensorFlow:", _e)
tf.get_logger().setLevel("ERROR")

warnings.filterwarnings('ignore')

SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__} | TensorFlow: {tf.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = "/kaggle/working/tfcwda_baseline"
MODELS_DIR = f"{OUTPUT_DIR}/models"
TABLES_DIR = f"{OUTPUT_DIR}/tables"
for d in [MODELS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- must match the im-main notebook exactly for an identical split ----
SOURCE_ALL = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
TARGET_ALL = "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset"

CLASSES = ["Tomato_Septoria_Leaf_Spot", "Potato_Early_Blight", "Corn_Leaf_Blight"]
NUM_CLASSES = len(CLASSES)

SOURCE_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato___Septoria_leaf_spot"],
    "Potato_Early_Blight":       ["Potato___Early_blight"],
    "Corn_Leaf_Blight":          ["Corn_(maize)___Northern_Leaf_Blight"],
}
TARGET_CLASS_DIRS = {
    "Tomato_Septoria_Leaf_Spot": ["Tomato Septoria leaf spot"],
    "Potato_Early_Blight":       ["Potato leaf early blight"],
    "Corn_Leaf_Blight":          ["Corn leaf blight"],
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ---- Table 2 of the paper ----
IMG_SIZE          = 224
BATCH_SIZE        = 32      # source batch (paper: 32)
TARGET_BATCH_SIZE = 32      # equal to the source batch
EPOCHS            = 50      # paper: 50
LR                = 3e-4    # paper: Adam, 3e-4
ROT_DEG           = 15      # paper: "dynamic horizontal flip, rotation" (angle not stated)

# ---- protocol shared with the other baselines ----
PATIENCE      = 15
SMOOTH_WINDOW = 5
SELECTION     = "smoothed_target_val"   # "last" = strict UDA, no target labels for selection
SEEDS         = [42, 1337, 2024]        # same as im-main and the EffNet baseline
PROBE_EPOCHS  = 2                       # short run used only to read peak GPU memory
RUN_ABLATION  = False                   # True -> also train the no-Wasserstein / no-UPL variant (Part 8)

# ---- paper components switches ----
TARGET_DENOISE        = True    # Gaussian + median filtering of PlantDoc images (Sec. 3.1)
CLASS_BALANCED_SAMPLE = True    # weighted random sampling of the source loader (Sec. 3.1)

# ---- model (ASSUMED sizes; the paper gives the layout of Fig. 4 only) ----
MV_CH     = (64, 96, 128)       # channels of the MobileViT branch at /8, /16, /32
MV_DIM    = (96, 120, 144)      # transformer width per stage (divisible by MV_HEADS)
MV_DEPTH  = (2, 3, 4)           # L = 2, 3, 4 (Fig. 4)
MV_HEADS  = 4
T_HEAD_CH = 256                 # width of the transformer-branch feature before its FC
DROPOUT   = 0.2

# ---- Wasserstein domain adaptation (ASSUMED; Sec. 3.3 gives the form, not the values) ----
LAMBDA_WD   = 0.1               # weight of the Wasserstein estimate (ramped 0 -> LAMBDA_WD)
LAMBDA_ALPHA= 10.0
N_CRITIC    = 5                 # critic steps per feature-extractor step
GP_GAMMA    = 10.0              # gradient-penalty weight (Gulrajani et al.)
LR_CRITIC   = 1e-4
CRITIC_HID  = 256
ALPHA_UPL   = 0.1               # weight of the uncertainty penalty loss
EN_L1       = 1e-7              # elastic net (Eq. 7)
EN_L2       = 1e-6
GRAD_CLIP   = 5.0

# ---- INT8 export: native TF/Keras rebuild -> TFLite full-integer PTQ (same route as im-main, NO ONNX) ----
RUN_INT8_EXPORT   = True        # False -> training + FP32 evaluation only (then no INT8 number exists)
EXPORT_SMOKE_TEST = True        # build + convert + run the export on an untrained net BEFORE training
CAL_N_SRC         = 400         # calibration: up to 400 source-VAL images ...
CAL_SEED          = 0           # ... plus ALL target-VAL images (test splits are never used for calibration)
INT8_EVAL_SRC     = True        # also run the INT8 model on the 564 source-test images (slower; False to skip)
TFLITE_THREADS    = 4
PARITY_N          = 32          # images used to check Keras-vs-PyTorch logits before converting
PARITY_FAIL_REL   = 0.05        # abort if max|logit diff| > 5% of the largest |logit|  (= weight-copy bug)
# Variants are tried in this order until one converts to full INT8. (gelu, attention):
#   gelu "exact" = erf GELU as trained; "tanh" = tanh-approximate GELU (max error ~1e-3)
#   attention "bmm" = BATCH_MATMUL; "decomp" = broadcast-mul + reduce-sum (no BATCH_MATMUL)
EXPORT_VARIANTS = [("exact", "bmm"), ("tanh", "bmm"), ("exact", "decomp"), ("tanh", "decomp")]

print(f"\nIMG={IMG_SIZE} | classes={CLASSES}")
print(f"epochs<={EPOCHS} patience={PATIENCE} selection={SELECTION} seeds={SEEDS}")
print(f"Adam lr={LR} | batch {BATCH_SIZE}/{TARGET_BATCH_SIZE} | critic steps={N_CRITIC} "
      f"| lambda<={LAMBDA_WD} alpha={ALPHA_UPL}")
print(f"INT8 export={RUN_INT8_EXPORT} | smoke test={EXPORT_SMOKE_TEST} | calibration = {CAL_N_SRC} src-val + all tgt-val")

Device: cuda | PyTorch: 2.11.0+cu128 | TensorFlow: 2.20.0
GPU: Tesla T4

IMG=224 | classes=['Tomato_Septoria_Leaf_Spot', 'Potato_Early_Blight', 'Corn_Leaf_Blight']
epochs<=50 patience=15 selection=smoothed_target_val seeds=[42, 1337, 2024]
Adam lr=0.0003 | batch 32/32 | critic steps=5 | lambda<=0.1 alpha=0.1
INT8 export=True | smoke test=True | calibration = 400 src-val + all tgt-val


## Part 1 - Data indexing and split (identical logic to im-main)

In [2]:
# ============================================================
# PART 1: DATASET INDEX + SPLIT  (identical to im-main / MSUN / EffNet notebooks)
# ============================================================

class LeafIndex:
    # Path/label index only. Walks root_dir recursively and pools every image under
    # any folder whose normalized name matches an alias for a class.

    def __init__(self, root_dir, classes, class_dir_map, domain_label=0, name=""):
        self.img_paths, self.labels = [], []
        self.classes, self.domain_label = classes, domain_label

        def _norm(s):
            return s.strip().lower().replace(" ", "").replace("_", "")

        for idx, cls in enumerate(classes):
            alias = {_norm(a) for a in class_dir_map.get(cls, [cls])}
            matched = [dp for dp, _, _ in os.walk(root_dir) if _norm(os.path.basename(dp)) in alias]
            for d in matched:
                for f in sorted(os.listdir(d)):
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                        self.img_paths.append(os.path.join(d, f)); self.labels.append(idx)
            if not matched:
                print(f"    WARNING: no folder matched '{cls}' under {root_dir}")

        print(f"  [{name}] {len(self.img_paths)} images")
        for i, c in enumerate(classes):
            print(f"    {c}: {self.labels.count(i)}")

    def subset(self, indices):
        s = copy.copy(self)
        s.img_paths = [self.img_paths[i] for i in indices]
        s.labels    = [self.labels[i] for i in indices]
        return s


def stratified_split_indices(labels, val_frac=0.15, test_frac=0.15, seed=SEED):
    idx_all = list(range(len(labels)))
    idx_tv, idx_test = train_test_split(idx_all, test_size=test_frac, stratify=labels,
                                        random_state=seed)
    lab_tv = [labels[i] for i in idx_tv]
    idx_tr, idx_val = train_test_split(idx_tv, test_size=val_frac / (1 - test_frac),
                                       stratify=lab_tv, random_state=seed)
    return idx_tr, idx_val, idx_test


for nm, p in [('PlantVillage color', SOURCE_ALL), ('PlantDoc', TARGET_ALL)]:
    print(f"  {nm:24s}: {'OK ' if os.path.isdir(p) else '!! NOT FOUND !!'} {p}")

print("\n--- Source (PlantVillage) ---")
src_index = LeafIndex(SOURCE_ALL, CLASSES, SOURCE_CLASS_DIRS, 0, "Source")
print("\n--- Target (PlantDoc) ---")
tgt_index = LeafIndex(TARGET_ALL, CLASSES, TARGET_CLASS_DIRS, 1, "Target")

src_tr_i, src_va_i, src_te_i = stratified_split_indices(src_index.labels)
tgt_tr_i, tgt_va_i, tgt_te_i = stratified_split_indices(tgt_index.labels)

src_train_idx = src_index.subset(src_tr_i)
src_val_idx   = src_index.subset(src_va_i)
src_test_idx  = src_index.subset(src_te_i)
tgt_train_idx = tgt_index.subset(tgt_tr_i)   # used UNLABELED in training (UDA)
tgt_val_idx   = tgt_index.subset(tgt_va_i)   # labeled, used only for checkpoint selection (+ INT8 calibration images)
tgt_test_idx  = tgt_index.subset(tgt_te_i)   # labeled, touched once at the very end

print(f"\nSource  train/val/test: {len(src_train_idx.labels)}/"
      f"{len(src_val_idx.labels)}/{len(src_test_idx.labels)}")
print(f"Target  train/val/test: {len(tgt_train_idx.labels)}/"
      f"{len(tgt_val_idx.labels)}/{len(tgt_test_idx.labels)}")
print(f"\nNOTE: target test = {len(tgt_test_idx.labels)} images -> 1 image = "
      f"{100/len(tgt_test_idx.labels):.2f} accuracy points. Report the bootstrap CI.")

  PlantVillage color      : OK  /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
  PlantDoc                : OK  /kaggle/input/datasets/nirmalsankalana/plantdoc-dataset

--- Source (PlantVillage) ---
  [Source] 3756 images
    Tomato_Septoria_Leaf_Spot: 1771
    Potato_Early_Blight: 1000
    Corn_Leaf_Blight: 985

--- Target (PlantDoc) ---
  [Target] 522 images
    Tomato_Septoria_Leaf_Spot: 157
    Potato_Early_Blight: 171
    Corn_Leaf_Blight: 194

Source  train/val/test: 2628/564/564
Target  train/val/test: 364/79/79

NOTE: target test = 79 images -> 1 image = 1.27 accuracy points. Report the bootstrap CI.


## Part 2 - Datasets and loaders

Training augmentation follows Table 2 (resize 224, horizontal flip, rotation). PlantDoc images additionally go through
median + Gaussian filtering (Sec. 3.1), applied to every target split. Source images are not filtered.
The source loader uses class-balanced weighted random sampling. The target-train loader returns labels too,
but the training loop never reads them.

In [3]:
# ============================================================
# PART 2: DATASETS AND LOADERS
# ============================================================

class DenoiseFilter:
    # Paper Sec. 3.1: Gaussian and median filtering to reduce PlantDoc noise.
    # Kernel size / sigma are not stated: median 3x3 and Gaussian radius 1 are assumed.
    def __call__(self, img):
        img = img.filter(ImageFilter.MedianFilter(size=3))
        return img.filter(ImageFilter.GaussianBlur(radius=1))


class LeafImageDataset(Dataset):
    def __init__(self, index_obj, transform):
        self.idx = index_obj
        self.transform = transform

    def __len__(self):
        return len(self.idx.img_paths)

    def __getitem__(self, i):
        img = Image.open(self.idx.img_paths[i]).convert("RGB")
        return self.transform(img), self.idx.labels[i]


def make_tf(train, denoise):
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if denoise:
        ops.append(DenoiseFilter())
    if train:
        ops += [transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(ROT_DEG)]
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)


src_train_ds = LeafImageDataset(src_train_idx, make_tf(True,  False))
src_val_ds   = LeafImageDataset(src_val_idx,   make_tf(False, False))
src_test_ds  = LeafImageDataset(src_test_idx,  make_tf(False, False))
tgt_train_ds = LeafImageDataset(tgt_train_idx, make_tf(True,  TARGET_DENOISE))  # labels never used in training
tgt_val_ds   = LeafImageDataset(tgt_val_idx,   make_tf(False, TARGET_DENOISE))
tgt_test_ds  = LeafImageDataset(tgt_test_idx,  make_tf(False, TARGET_DENOISE))


def mk_loader(ds, bs, shuffle, drop_last=False, sampler=None):
    return DataLoader(ds, batch_size=bs, shuffle=(shuffle and sampler is None), sampler=sampler,
                      num_workers=4, pin_memory=True, drop_last=drop_last, persistent_workers=True)


# class-balanced weighted random sampling of the source domain (paper Sec. 3.1)
_cnt = np.bincount(src_train_idx.labels, minlength=NUM_CLASSES)
_w = 1.0 / _cnt[np.array(src_train_idx.labels)]
src_sampler = (WeightedRandomSampler(torch.DoubleTensor(_w), num_samples=len(_w), replacement=True)
               if CLASS_BALANCED_SAMPLE else None)

# drop_last on the source loader avoids a tiny last batch in BatchNorm train mode
src_train_loader = mk_loader(src_train_ds, BATCH_SIZE, True, drop_last=True, sampler=src_sampler)
tgt_train_loader = mk_loader(tgt_train_ds, TARGET_BATCH_SIZE, True, drop_last=False)
src_val_loader   = mk_loader(src_val_ds,   BATCH_SIZE, False)
src_test_loader  = mk_loader(src_test_ds,  BATCH_SIZE, False)
tgt_val_loader   = mk_loader(tgt_val_ds,   BATCH_SIZE, False)
tgt_test_loader  = mk_loader(tgt_test_ds,  BATCH_SIZE, False)

def cycle(loader):
    while True:
        for batch in loader:
            yield batch

print(f"source class counts (train)  : {_cnt.tolist()}  balanced sampling={CLASS_BALANCED_SAMPLE}")
print(f"source train batches/epoch   : {len(src_train_loader)}")
print(f"target train batches/epoch   : {len(tgt_train_loader)} (cycled, unlabeled use)")
xb, yb = next(iter(src_train_loader))
print(f"batch shape: {tuple(xb.shape)}  label sample: {yb[:8].tolist()}")

# Start every persistent worker pool NOW, before TensorFlow executes any op (Part 6b / 7).
# Forking DataLoader workers from a process whose TF runtime is already running can hang; this avoids it.
for _ld in (tgt_train_loader, src_val_loader, src_test_loader, tgt_val_loader, tgt_test_loader):
    next(iter(_ld))
print("all DataLoader worker pools started")

source class counts (train)  : [1239, 700, 689]  balanced sampling=True
source train batches/epoch   : 82
target train batches/epoch   : 12 (cycled, unlabeled use)
batch shape: (32, 3, 224, 224)  label sample: [0, 0, 1, 0, 0, 1, 0, 0]
all DataLoader worker pools started


## Part 3 - Transformer-fused convolution network (Fig. 4 and Sec. 3.2)

Layout, with a 224 px input:

| stage | CNN branch (MobileNetV2, pretrained) | MobileViT branch | grid |
|---|---|---|---|
| 1 | `features[:7]`, 32 ch | 1x1 align, MobileViT block L=2 | 28x28 |
| 2 | `features[7:14]`, 96 ch | stride-2 conv + aligned CNN feature, MobileViT block L=3 | 14x14 |
| 3 | `features[14:18]`, 320 ch | stride-2 conv + aligned CNN feature, MobileViT block L=4 | 7x7 |

After every MobileViT block its output is projected back (1x1 conv) and added to the CNN feature, so each branch
receives the other's information. Each branch ends in global pooling and its own FC layer. The concatenated pooled
features (1280 + 256) are the representation `h = f_g(x)` that the Wasserstein critic sees.
The MobileViT block follows Sec. 3.2.1: n x n conv, 1x1 conv to dimension d, unfold into 2x2 patch positions,
transformer across patches, fold, 1x1 conv back to C, concatenate with the input, n x n conv.

In [4]:
# ============================================================
# PART 3: MODEL
# ============================================================

class ConvBNAct(nn.Sequential):
    def __init__(self, cin, cout, k=1, s=1, act=True):
        layers = [nn.Conv2d(cin, cout, k, s, k // 2, bias=False), nn.BatchNorm2d(cout)]
        if act:
            layers.append(nn.SiLU(inplace=True))
        super().__init__(*layers)


class MobileViTBlock(nn.Module):
    # Sec. 3.2.1: local conv -> unfold -> transformer -> fold -> project -> concat -> conv.
    def __init__(self, ch, dim, depth, heads, patch=2, ff_mult=2):
        super().__init__()
        self.patch = patch
        self.local = nn.Sequential(ConvBNAct(ch, ch, 3), ConvBNAct(ch, dim, 1))
        layer = nn.TransformerEncoderLayer(dim, heads, dim * ff_mult, dropout=0.0,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.proj = ConvBNAct(dim, ch, 1)
        self.fuse = ConvBNAct(2 * ch, ch, 3)

    def forward(self, x):
        res = x
        y = self.local(x)
        B, D, H, W = y.shape
        p = self.patch
        nH, nW = math.ceil(H / p) * p, math.ceil(W / p) * p       # 7x7 -> 8x8
        resized = (nH, nW) != (H, W)
        if resized:
            y = F.interpolate(y, size=(nH, nW), mode="bilinear", align_corners=False)
        h, w = nH // p, nW // p
        # (B,D,H,W) -> (B*p*p, h*w, D): attention runs across patches, per pixel position
        y = y.reshape(B, D, h, p, w, p).permute(0, 3, 5, 2, 4, 1).reshape(B * p * p, h * w, D)
        y = self.norm(self.transformer(y))
        y = y.reshape(B, p, p, h, w, D).permute(0, 5, 3, 1, 4, 2).reshape(B, D, nH, nW)
        if resized:
            y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        y = self.proj(y)
        return self.fuse(torch.cat([res, y], dim=1))


def build_mobilenet_v2():
    try:
        return tvm.mobilenet_v2(weights=tvm.MobileNet_V2_Weights.IMAGENET1K_V1)
    except Exception:
        return tvm.mobilenet_v2(pretrained=True)


class TFCNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        f = build_mobilenet_v2().features
        self.cnn1, self.cnn2, self.cnn3, self.cnn_last = f[:7], f[7:14], f[14:18], f[18]
        cch = (32, 96, 320)

        # feature fusing: align channel dimensions with 1x1 convs (+ BN), both directions
        self.c2t = nn.ModuleList([ConvBNAct(c, d, 1) for c, d in zip(cch, MV_CH)])
        self.t2c = nn.ModuleList([ConvBNAct(d, c, 1, act=False) for c, d in zip(cch, MV_CH)])
        self.t_down = nn.ModuleList([ConvBNAct(MV_CH[0], MV_CH[1], 3, 2),
                                     ConvBNAct(MV_CH[1], MV_CH[2], 3, 2)])
        self.mvit = nn.ModuleList([MobileViTBlock(c, d, L, MV_HEADS)
                                   for c, d, L in zip(MV_CH, MV_DIM, MV_DEPTH)])
        self.t_head = ConvBNAct(MV_CH[2], T_HEAD_CH, 1)

        self.drop = nn.Dropout(DROPOUT)
        self.fc_c = nn.Linear(1280, num_classes)          # CNN-branch classifier
        self.fc_t = nn.Linear(T_HEAD_CH, num_classes)     # transformer-branch classifier
        self.feat_dim = 1280 + T_HEAD_CH

    def forward(self, x, return_all=False):
        c1 = self.cnn1(x)                                  # (B, 32, 28, 28)
        t = self.mvit[0](self.c2t[0](c1))
        c1 = c1 + self.t2c[0](t)

        c2 = self.cnn2(c1)                                 # (B, 96, 14, 14)
        t = self.t_down[0](t) + self.c2t[1](c2)
        t = self.mvit[1](t)
        c2 = c2 + self.t2c[1](t)

        c3 = self.cnn3(c2)                                 # (B, 320, 7, 7)
        t = self.t_down[1](t) + self.c2t[2](c3)
        t = self.mvit[2](t)
        c3 = c3 + self.t2c[2](t)

        fc = F.adaptive_avg_pool2d(self.cnn_last(c3), 1).flatten(1)
        ft = F.adaptive_avg_pool2d(self.t_head(t), 1).flatten(1)
        lc, lt = self.fc_c(self.drop(fc)), self.fc_t(self.drop(ft))
        logits = 0.5 * (lc + lt)
        if return_all:
            return logits, lc, lt, torch.cat([fc, ft], dim=1)
        return logits


class DomainCritic(nn.Module):
    # f_w : R^d -> R. No normalization layers (WGAN-GP).
    def __init__(self, d, hidden=CRITIC_HID):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(True),
                                 nn.Linear(hidden, hidden), nn.ReLU(True),
                                 nn.Linear(hidden, 1))

    def forward(self, h):
        return self.net(h).squeeze(1)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


_probe = TFCNet(NUM_CLASSES)
_n = count_params(_probe)
print(f"TFC classifier: {_n:,} params | FP32 ~= {_n*4/1024**2:.2f} MB  (critic is training-only, not counted)")
with torch.no_grad():
    _lg, _lc, _lt, _h = _probe.eval()(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE), return_all=True)
print(f"logits {tuple(_lg.shape)} | branch logits {tuple(_lc.shape)}, {tuple(_lt.shape)} | feature h {tuple(_h.shape)}")
del _probe, _lg, _lc, _lt, _h

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 109MB/s] 


TFC classifier: 4,580,190 params | FP32 ~= 17.47 MB  (critic is training-only, not counted)
logits (2, 3) | branch logits (2, 3), (2, 3) | feature h (2, 1536)


## Part 4 - Losses and utilities

- **Critic loss** (Eqs. 9-11): `L_wd = mean f_w(h_s) - mean f_w(h_t)`, gradient penalty on random interpolates (Eq. 10). The critic maximizes `L_wd - gamma * L_grad`.
- **Feature extractor** (Eq. 12): minimizes `L_wd` with the critic frozen. The gradient penalty is dropped there (gamma = 0), as the paper states.
- **UPL** (Eq. 14, Guided Complement Entropy): for each source sample, the wrong-class probabilities are renormalized and their entropy is computed.
  The paper's motivation (the flat `[0.5, 0.35, 0.35]` is preferred over `[0.5, 0.4, 0.3]`) means this entropy must be *maximized*, so the loss minimized here is its negative.
- **Elastic net** (Eq. 7) over all weight matrices and kernels.

In [5]:
# ============================================================
# PART 4: LOSSES AND UTILITIES
# ============================================================

def gradient_penalty(critic, hs, ht):
    # Eq. 10: (||grad f_w(h_hat)||_2 - 1)^2 on random points between source / target features
    n = min(hs.size(0), ht.size(0))
    hs, ht = hs[:n], ht[:n]
    eps = torch.rand(n, 1, device=hs.device)
    hat = (eps * hs + (1 - eps) * ht).requires_grad_(True)
    out = critic(hat)
    g = torch.autograd.grad(out.sum(), hat, create_graph=True)[0]
    return ((g.norm(2, dim=1) - 1) ** 2).mean()


def upl_loss(logits, y):
    # Eq. 14: entropy of the renormalized wrong-class distribution, returned NEGATED so that
    # minimizing it flattens the wrong-class probabilities (see markdown above and caveats).
    p = F.softmax(logits, dim=1)
    wrong = p.masked_fill(F.one_hot(y, logits.size(1)).bool(), 0.0)
    wrong = wrong / (wrong.sum(1, keepdim=True) + 1e-8)
    ent = -(wrong * torch.log(wrong + 1e-8)).sum(1)
    return -ent.mean()


def elastic_net_penalty(model):
    # Eq. 7: lambda1 * ||W||_1 + lambda2 * ||W||_2^2 over weight tensors (dim > 1)
    l1, l2 = 0.0, 0.0
    for p in model.parameters():
        if p.requires_grad and p.dim() > 1:
            l1 = l1 + p.abs().sum()
            l2 = l2 + p.pow(2).sum()
    return EN_L1 * l1 + EN_L2 * l2


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-3):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best, self.early_stop = 0, None, False

    def __call__(self, score):
        if self.best is None or score > self.best + self.min_delta:
            self.best, self.counter = score, 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True
        return False


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    tot, preds, labs = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        tot += criterion(logits, y).item() * x.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(labs)
    return (tot / n, accuracy_score(labs, preds),
            f1_score(labs, preds, average='weighted'),
            np.array(preds), np.array(labs))


def full_metrics(labels, preds, num_classes=NUM_CLASSES):
    # Metrics of the paper's Sec. 3.5. "bal_acc_paper" = macro mean of (TPR + TNR) / 2 (Eq. 16f),
    # "bal_acc_sklearn" = mean recall. They differ, so both are reported.
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    tp = np.diag(cm).astype(float)
    fn = cm.sum(1) - tp
    fp = cm.sum(0) - tp
    tn = cm.sum() - tp - fn - fp
    tpr, tnr = tp / (tp + fn + 1e-12), tn / (tn + fp + 1e-12)
    return dict(
        acc=accuracy_score(labels, preds),
        bal_acc_paper=float(((tpr + tnr) / 2).mean()),
        bal_acc_sklearn=balanced_accuracy_score(labels, preds),
        prec_macro=precision_score(labels, preds, average='macro', zero_division=0),
        prec_micro=precision_score(labels, preds, average='micro', zero_division=0),
        rec_macro=recall_score(labels, preds, average='macro', zero_division=0),
        rec_micro=recall_score(labels, preds, average='micro', zero_division=0),
        f1_macro=f1_score(labels, preds, average='macro', zero_division=0),
        f1_micro=f1_score(labels, preds, average='micro', zero_division=0),
        f1_weighted=f1_score(labels, preds, average='weighted', zero_division=0),
        kappa=cohen_kappa_score(labels, preds),
    )


def bootstrap_ci(labels, preds, n_boot=5000, seed=SEED):
    rng = np.random.RandomState(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    accs = np.array([(labels[i] == preds[i]).mean()
                     for i in (rng.randint(0, n, n) for _ in range(n_boot))])
    return accs.mean(), *np.percentile(accs, [2.5, 97.5])


cls_criterion = nn.CrossEntropyLoss()
print("Utilities ready: gradient_penalty, upl_loss, elastic_net_penalty, EarlyStopping, evaluate, full_metrics, bootstrap_ci")

Utilities ready: gradient_penalty, upl_loss, elastic_net_penalty, EarlyStopping, evaluate, full_metrics, bootstrap_ci


## Part 5 - One training run

Each iteration takes one source batch and one target batch and forwards them in separate passes.
1. The critic is updated `N_CRITIC` times on the detached features (maximize `L_wd - gamma * L_grad`).
2. The feature extractor and both classifiers are updated with
   `CE(branch C) + CE(branch T) + lambda * W + alpha * UPL + elastic net`, the critic frozen.

lambda ramps with `2 / (1 + exp(-10 p)) - 1`. Checkpoint selection mirrors the other baselines: 5-epoch moving average of
target-val accuracy, patience 15. Set `SELECTION = "last"` for strict UDA, which never looks at target labels.
`use_da=False` removes the Wasserstein and UPL terms (ablation).

In [6]:
# ============================================================
# PART 5: TRAIN ONE RUN
# ============================================================

def train_one_run(seed, epochs=EPOCHS, selection=SELECTION, use_da=True, verbose=True, tag="run"):
    set_seed(seed)
    model = TFCNet(NUM_CLASSES).to(DEVICE)
    critic = DomainCritic(model.feat_dim).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    opt_c = optim.Adam(critic.parameters(), lr=LR_CRITIC, betas=(0.5, 0.9))
    stopper = EarlyStopping(patience=PATIENCE)

    total_iters = epochs * len(src_train_loader)
    tgt_iter = cycle(tgt_train_loader)
    step = 0
    raw_hist, best_sm = [], 0.0
    best_state = copy.deepcopy(model.state_dict())
    t0 = time.time()

    if verbose:
        print("=" * 100)
        print(f"  [{tag}] seed={seed} | epochs<={epochs} | selection={selection} | DA={'ON' if use_da else 'OFF'}")
        print("=" * 100)

    for ep in range(1, epochs + 1):
        model.train()
        n_seen, n_correct = 0, 0
        s_ce = s_w = s_upl = 0.0
        for xs, ys in src_train_loader:
            p = step / max(total_iters, 1)
            lam = LAMBDA_WD * (2.0 / (1.0 + math.exp(-LAMBDA_ALPHA * p)) - 1.0)

            xt, _ = next(tgt_iter)                    # target labels are NOT used
            xs, ys, xt = xs.to(DEVICE), ys.to(DEVICE), xt.to(DEVICE)

            logits_s, lc_s, lt_s, hs = model(xs, return_all=True)
            ce = F.cross_entropy(lc_s, ys) + F.cross_entropy(lt_s, ys)
            loss = ce + elastic_net_penalty(model)
            w_est = upl = torch.zeros((), device=DEVICE)

            if use_da:
                _, _, _, ht = model(xt, return_all=True)

                # (1) critic: maximize L_wd - gamma * L_grad on detached features (Eq. 11)
                hs_d, ht_d = hs.detach(), ht.detach()
                for _ in range(N_CRITIC):
                    l_wd = critic(hs_d).mean() - critic(ht_d).mean()
                    gp = gradient_penalty(critic, hs_d, ht_d)
                    opt_c.zero_grad(set_to_none=True)
                    (-(l_wd - GP_GAMMA * gp)).backward()
                    opt_c.step()

                # (2) feature extractor: minimize the Wasserstein estimate, critic frozen, gamma = 0 (Eq. 12)
                for q in critic.parameters():
                    q.requires_grad_(False)
                w_est = critic(hs).mean() - critic(ht).mean()
                for q in critic.parameters():
                    q.requires_grad_(True)

                upl = upl_loss(logits_s, ys)
                loss = loss + lam * w_est + ALPHA_UPL * upl

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            step += 1

            n_correct += (logits_s.argmax(1) == ys).sum().item()
            n_seen += ys.size(0)
            s_ce += ce.item() * ys.size(0)
            s_w += float(w_est) * ys.size(0)
            s_upl += float(upl) * ys.size(0)

        _, sv_acc, _, _, _     = evaluate(model, src_val_loader, cls_criterion, DEVICE)
        _, tv_acc, tv_f1, _, _ = evaluate(model, tgt_val_loader, cls_criterion, DEVICE)
        raw_hist.append(tv_acc)
        sm = float(np.mean(raw_hist[-SMOOTH_WINDOW:]))

        if selection == "smoothed_target_val":
            stopper(sm)
            if sm > best_sm:
                best_sm = sm
                best_state = copy.deepcopy(model.state_dict())

        if verbose and (ep % 5 == 0 or ep == 1):
            print(f"Ep {ep:3d}/{epochs} | ce={s_ce/n_seen:.3f} W={s_w/n_seen:.4f} upl={s_upl/n_seen:.3f} "
                  f"lam={lam:.3f} | train_acc={n_correct/n_seen:.3f} | SRC-VAL={sv_acc:.3f} | "
                  f"TGT-VAL={tv_acc:.3f} (sm={sm:.3f}) f1={tv_f1:.3f} | {time.time()-t0:.0f}s")

        if selection == "smoothed_target_val" and stopper.early_stop:
            if verbose:
                print(f"\nEarly stop at epoch {ep}")
            break

    if selection == "smoothed_target_val":
        model.load_state_dict(best_state)
    train_seconds = time.time() - t0
    if verbose:
        print(f"Done in {train_seconds:.0f}s | best smoothed TGT-VAL = {best_sm:.4f}")
    return model, raw_hist, best_sm, train_seconds

## Part 6 - Real INT8 export machinery (PyTorch -> native TF/Keras -> TFLite full-integer PTQ)

This is the same route as im-main Part 10B, **without ONNX**: the trained PyTorch network is rebuilt layer by layer in
TF/Keras, every weight tensor is copied from the PyTorch module it belongs to (shape-checked by `set_weights`), and the
Keras model is quantized by the standard `tf.lite.TFLiteConverter` with a representative dataset.

**Raw data path.** The TF model takes raw RGB in [0, 1] and applies the ImageNet mean/std *inside the graph*, exactly like
im-main bakes in its `(x - 0.5) / 0.5`. Raw uint8 arrays are produced with the same `Resize -> (denoise)` pipeline as the
evaluation loaders, so the PyTorch FP32 model, the Keras model, the FP32 TFLite and the INT8 TFLite all see the same pixels.

**Translation details (all checked numerically by the parity gate before conversion).**
- Stride-2 3x3 convs use explicit `ZeroPadding2D(1)` + `valid` (TF `same` pads asymmetrically and would not match PyTorch).
- BatchNorm is kept as BN (epsilon from the PyTorch module); the converter folds it into the conv weights before quantizing.
- SiLU = `x * sigmoid(x)`. The two branch classifiers are folded into **one** FC on `[fc, ft]` with weights `0.5*[Wc, Wt]`, bias `0.5*(bc+bt)` (exact).
- MobileViT unfold/fold is done with 4 strided slices + a batch concat and a stack/reshape interleave (<= 5-D tensors, no 6-D transpose).
- The 1/sqrt(d_head) attention scale is folded into the Q projection (exact). The 7x7 -> 8x8 -> 7x7 bilinear resize uses half-pixel centers = PyTorch `align_corners=False`.
- GELU / attention variants are tried in the order of `EXPORT_VARIANTS` until one converts to **full INT8** (`TFLITE_BUILTINS_INT8`, int8 input and output, so no float fallback op can hide in the file). The variant used is stored per seed.

**What "real INT8" means here, and how it is verified.** Every run prints an audit of the final `.tflite`: input/output dtype and
quantization parameters, a histogram of tensor dtypes, the op list, and any float32 tensor that is not a tiny scalar. The reported size is `os.path.getsize` of the file.

In [7]:
# ============================================================
# PART 6A: RAW UINT8 ARRAYS (same pixels for PyTorch, Keras and TFLite) + PYTORCH LOGITS FROM RAW
# ============================================================
IMNET_MEAN_NP = np.array(IMAGENET_MEAN, np.float32)
IMNET_STD_NP  = np.array(IMAGENET_STD,  np.float32)


def load_raw_arrays(idx_obj, denoise, name=""):
    # Same ops as make_tf(train=False, denoise) minus ToTensor/Normalize. ToTensor is x/255, so uint8 is lossless.
    rs = transforms.Resize((IMG_SIZE, IMG_SIZE))
    dn = DenoiseFilter() if denoise else None
    arr = np.empty((len(idx_obj.img_paths), IMG_SIZE, IMG_SIZE, 3), np.uint8)
    for i, pth in enumerate(idx_obj.img_paths):
        img = rs(Image.open(pth).convert("RGB"))
        if dn is not None:
            img = dn(img)
        arr[i] = np.asarray(img, dtype=np.uint8)
    print(f"  raw[{name:8s}] {arr.shape}  {arr.nbytes/1024**2:6.1f} MB  denoise={denoise}")
    return arr


RAW = {}
if RUN_INT8_EXPORT:
    print("Loading raw uint8 arrays (one time) ...")
    RAW["src_val"],  RAW["src_val_labels"]  = load_raw_arrays(src_val_idx,  False, "src_val"),  np.array(src_val_idx.labels)
    RAW["src_test"], RAW["src_test_labels"] = load_raw_arrays(src_test_idx, False, "src_test"), np.array(src_test_idx.labels)
    RAW["tgt_val"],  RAW["tgt_val_labels"]  = load_raw_arrays(tgt_val_idx,  TARGET_DENOISE, "tgt_val"),  np.array(tgt_val_idx.labels)
    RAW["tgt_test"], RAW["tgt_test_labels"] = load_raw_arrays(tgt_test_idx, TARGET_DENOISE, "tgt_test"), np.array(tgt_test_idx.labels)


@torch.no_grad()
def torch_logits_from_raw(model, arr_u8, device="cpu", bs=32):
    model.eval()
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
    outs = []
    for i in range(0, len(arr_u8), bs):
        x = torch.from_numpy(arr_u8[i:i + bs]).to(device).permute(0, 3, 1, 2).float() / 255.0
        outs.append(model((x - mean) / std).float().cpu())
    return torch.cat(outs).numpy()


def make_calibration(n_src, seed=CAL_SEED):
    # Representative set for the quantizer: source-VAL (subsampled) + ALL target-VAL. No test image is used.
    rng = np.random.RandomState(seed)
    pick = rng.permutation(len(RAW["src_val"]))[:n_src]
    cal = np.concatenate([RAW["src_val"][pick], RAW["tgt_val"]], axis=0)
    return cal[rng.permutation(len(cal))]


if RUN_INT8_EXPORT:
    _c = make_calibration(CAL_N_SRC)
    print(f"Calibration set: {_c.shape[0]} images ({min(CAL_N_SRC, len(RAW['src_val']))} source-val + {len(RAW['tgt_val'])} target-val), "
          f"raw uint8 range [{_c.min()}, {_c.max()}]")
    del _c

Loading raw uint8 arrays (one time) ...
  raw[src_val ] (564, 224, 224, 3)    81.0 MB  denoise=False
  raw[src_test] (564, 224, 224, 3)    81.0 MB  denoise=False
  raw[tgt_val ] (79, 224, 224, 3)    11.3 MB  denoise=True
  raw[tgt_test] (79, 224, 224, 3)    11.3 MB  denoise=True
Calibration set: 479 images (400 source-val + 79 target-val), raw uint8 range [0, 255]


In [8]:
# ============================================================
# PART 6B: TORCH MODULE -> KERAS LAYERS (weights copied module by module)
# ============================================================
L = tf.keras.layers
_uid = {"n": 0}

def _nm(prefix):
    _uid["n"] += 1
    return f"{prefix}_{_uid['n']}"

def _np(t):
    return t.detach().cpu().numpy().astype(np.float32)


def _silu(x, name):
    s = L.Activation("sigmoid", name=_nm(name + "_sig"))(x)
    return L.Multiply(name=_nm(name + "_mul"))([x, s])


def _conv(x, m, name):
    kh, kw = m.kernel_size
    assert kh == kw and tuple(m.dilation) == (1, 1), "only square, non-dilated convs are supported"
    k, s, pad, g = kh, m.stride[0], m.padding[0], m.groups
    assert pad in (0, k // 2), f"unsupported padding {pad} for k={k}"
    dw = g > 1
    if dw:
        assert g == m.in_channels == m.out_channels, "only depthwise grouped convs are supported"
    if s > 1 and pad > 0:                      # explicit symmetric pad, as PyTorch (TF 'same' is asymmetric)
        x = L.ZeroPadding2D(pad, name=_nm(name + "_pad"))(x)
        padding = "valid"
    else:
        padding = "same" if pad > 0 else "valid"
    use_bias = m.bias is not None
    W = _np(m.weight)
    if dw:
        layer = L.DepthwiseConv2D(k, strides=s, padding=padding, depth_multiplier=1, use_bias=use_bias, name=_nm(name))
        y = layer(x)
        ws = [W.transpose(2, 3, 0, 1)]         # (C,1,kh,kw) -> (kh,kw,C,1)
    else:
        layer = L.Conv2D(m.out_channels, k, strides=s, padding=padding, use_bias=use_bias, name=_nm(name))
        y = layer(x)
        ws = [W.transpose(2, 3, 1, 0)]         # (Cout,Cin,kh,kw) -> (kh,kw,Cin,Cout)
    if use_bias:
        ws.append(_np(m.bias))
    layer.set_weights(ws)
    return y


def _bn(x, m, name):
    layer = L.BatchNormalization(epsilon=m.eps, name=_nm(name))
    y = layer(x)
    layer.set_weights([_np(m.weight), _np(m.bias), _np(m.running_mean), _np(m.running_var)])
    return y


def _ln(x, m, name):
    layer = L.LayerNormalization(axis=-1, epsilon=m.eps, name=_nm(name))
    y = layer(x)
    layer.set_weights([_np(m.weight), _np(m.bias)])
    return y


def _dense(x, W_out_in, b, name):
    layer = L.Dense(W_out_in.shape[0], name=_nm(name))
    y = layer(x)
    layer.set_weights([np.ascontiguousarray(W_out_in.T), np.ascontiguousarray(b)])
    return y


def _tr(x, m, name):
    # Generic translator for the module types that occur in MobileNetV2 and in ConvBNAct.
    cn = m.__class__.__name__
    if isinstance(m, nn.Conv2d):
        return _conv(x, m, name)
    if isinstance(m, nn.BatchNorm2d):
        return _bn(x, m, name)
    if isinstance(m, nn.ReLU6):
        return L.ReLU(max_value=6.0, name=_nm(name))(x)
    if isinstance(m, nn.ReLU):
        return L.ReLU(name=_nm(name))(x)
    if isinstance(m, nn.SiLU):
        return _silu(x, name)
    if isinstance(m, (nn.Dropout, nn.Identity)):
        return x
    if cn == "InvertedResidual":
        y = _tr(x, m.conv, name)
        return L.Add(name=_nm(name + "_res"))([x, y]) if m.use_res_connect else y
    if isinstance(m, nn.Sequential):
        for i, c in enumerate(m):
            x = _tr(x, c, f"{name}_{i}")
        return x
    raise NotImplementedError(f"no Keras translation for {cn}")


def _gelu(x, mode, name):
    approx = (mode == "tanh")
    return L.Lambda(lambda t: tf.nn.gelu(t, approximate=approx), name=_nm(name + "_gelu"))(x)


def _mha(x, att, N, D, nh, attn_impl, name):
    # nn.MultiheadAttention (batch_first): in_proj = [Wq; Wk; Wv], softmax attention over the N tokens.
    dh = D // nh
    Wi, bi = _np(att.in_proj_weight), _np(att.in_proj_bias)
    scale = 1.0 / math.sqrt(dh)
    q = _dense(x, Wi[:D] * scale, bi[:D] * scale, name + "_q")      # attention scale folded into Q (exact)
    k = _dense(x, Wi[D:2 * D], bi[D:2 * D], name + "_k")
    v = _dense(x, Wi[2 * D:], bi[2 * D:], name + "_v")

    def heads(t, tag):
        t = L.Reshape((N, nh, dh), name=_nm(name + tag + "_r"))(t)
        return L.Permute((2, 1, 3), name=_nm(name + tag + "_p"))(t)          # (G, nh, N, dh)

    qh, kh, vh = heads(q, "_qh"), heads(k, "_kh"), heads(v, "_vh")
    if attn_impl == "bmm":
        sc = L.Lambda(lambda a: tf.matmul(a[0], a[1], transpose_b=True), name=_nm(name + "_qk"))([qh, kh])
        pr = L.Softmax(axis=-1, name=_nm(name + "_softmax"))(sc)
        cx = L.Lambda(lambda a: tf.matmul(a[0], a[1]), name=_nm(name + "_pv"))([pr, vh])
    elif attn_impl == "decomp":
        sc = L.Lambda(lambda a: tf.reduce_sum(tf.expand_dims(a[0], 3) * tf.expand_dims(a[1], 2), axis=-1),
                      name=_nm(name + "_qk"))([qh, kh])
        pr = L.Softmax(axis=-1, name=_nm(name + "_softmax"))(sc)
        cx = L.Lambda(lambda a: tf.reduce_sum(tf.expand_dims(a[0], -1) * tf.expand_dims(a[1], 2), axis=3),
                      name=_nm(name + "_pv"))([pr, vh])
    else:
        raise ValueError(attn_impl)
    cx = L.Permute((2, 1, 3), name=_nm(name + "_cp"))(cx)                    # (G, N, nh, dh)
    cx = L.Reshape((N, D), name=_nm(name + "_cr"))(cx)
    return _dense(cx, _np(att.out_proj.weight), _np(att.out_proj.bias), name + "_o")


def _enc_layer(x, lyr, N, D, nh, gelu_mode, attn_impl, name):
    # nn.TransformerEncoderLayer(norm_first=True, activation="gelu", dropout=0)
    assert getattr(lyr, "norm_first", False), "translator expects pre-norm encoder layers"
    a = _mha(_ln(x, lyr.norm1, name + "_n1"), lyr.self_attn, N, D, nh, attn_impl, name + "_sa")
    x = L.Add(name=_nm(name + "_r1"))([x, a])
    f = _dense(_ln(x, lyr.norm2, name + "_n2"), _np(lyr.linear1.weight), _np(lyr.linear1.bias), name + "_ff1")
    f = _gelu(f, gelu_mode, name)
    f = _dense(f, _np(lyr.linear2.weight), _np(lyr.linear2.bias), name + "_ff2")
    return L.Add(name=_nm(name + "_r2"))([x, f])


def _slicer(i, j, p):
    return lambda t: t[:, i::p, j::p, :]

def _gslice(g):
    return lambda t: t[g:g + 1]


def _mvit(x, tb, gelu_mode, attn_impl, name):
    res = x
    y = _tr(x, tb.local, name + "_loc")
    H, W, D = int(y.shape[1]), int(y.shape[2]), int(y.shape[3])
    p = tb.patch
    nH, nW = math.ceil(H / p) * p, math.ceil(W / p) * p
    resized = (nH, nW) != (H, W)
    if resized:                                                           # 7x7 -> 8x8, half-pixel = align_corners=False
        y = L.Lambda(lambda t: tf.image.resize(t, (nH, nW), method="bilinear"), name=_nm(name + "_up"))(y)
    h, w = nH // p, nW // p
    N, G = h * w, p * p
    # unfold: G pixel-offset groups, each (1, N, D); batched into (G, N, D). Same ordering as the PyTorch permute.
    parts = []
    for i in range(p):
        for j in range(p):
            s = L.Lambda(_slicer(i, j, p), name=_nm(name + "_sl"))(y)
            parts.append(L.Reshape((N, D), name=_nm(name + "_slr"))(s))
    tok = L.Lambda(lambda a: tf.concat(a, axis=0), name=_nm(name + "_cat"))(parts)
    nh = tb.transformer.layers[0].self_attn.num_heads
    for li, lyr in enumerate(tb.transformer.layers):
        tok = _enc_layer(tok, lyr, N, D, nh, gelu_mode, attn_impl, f"{name}_enc{li}")
    tok = _ln(tok, tb.norm, name + "_tnorm")
    # fold: undo the strided slices with stack+reshape interleaves
    grid = {}
    for g in range(G):
        i, j = divmod(g, p)
        t = L.Lambda(_gslice(g), name=_nm(name + "_gs"))(tok)
        grid[(i, j)] = L.Reshape((h, w, D), name=_nm(name + "_gr"))(t)
    rows = []
    for i in range(p):
        cols = [grid[(i, j)] for j in range(p)]
        rows.append(L.Lambda(lambda a: tf.reshape(tf.stack(a, axis=3), (1, h, p * w, D)), name=_nm(name + "_il"))(cols))
    y = L.Lambda(lambda a: tf.reshape(tf.stack(a, axis=2), (1, p * h, p * w, D)), name=_nm(name + "_il2"))(rows)
    if resized:                                                           # 8x8 -> 7x7
        y = L.Lambda(lambda t: tf.image.resize(t, (H, W), method="bilinear"), name=_nm(name + "_down"))(y)
    y = _tr(y, tb.proj, name + "_proj")
    cat = L.Concatenate(axis=-1, name=_nm(name + "_ccat"))([res, y])
    return _tr(cat, tb.fuse, name + "_fuse")


def build_keras_tfc(tm, img_size, gelu_mode, attn_impl):
    # tm: trained TFCNet on CPU in eval mode. Returns a Keras model: raw RGB [0,1] (1,S,S,3) -> logits (1,C).
    _uid["n"] = 0
    tm = tm.cpu().eval()
    inp = L.Input(shape=(img_size, img_size, 3), batch_size=1, name="input")
    sc = (1.0 / IMNET_STD_NP).astype(np.float32)
    of = (-IMNET_MEAN_NP * sc).astype(np.float32)
    x = L.Lambda(lambda t: t * sc + of, name="input_norm")(inp)          # (x - mean) / std baked into the graph

    c1 = _tr(x, tm.cnn1, "cnn1")
    t = _mvit(_tr(c1, tm.c2t[0], "c2t0"), tm.mvit[0], gelu_mode, attn_impl, "mv0")
    c1 = L.Add(name=_nm("fuse1"))([c1, _tr(t, tm.t2c[0], "t2c0")])

    c2 = _tr(c1, tm.cnn2, "cnn2")
    t = L.Add(name=_nm("tfuse2"))([_tr(t, tm.t_down[0], "tdown0"), _tr(c2, tm.c2t[1], "c2t1")])
    t = _mvit(t, tm.mvit[1], gelu_mode, attn_impl, "mv1")
    c2 = L.Add(name=_nm("fuse2"))([c2, _tr(t, tm.t2c[1], "t2c1")])

    c3 = _tr(c2, tm.cnn3, "cnn3")
    t = L.Add(name=_nm("tfuse3"))([_tr(t, tm.t_down[1], "tdown1"), _tr(c3, tm.c2t[2], "c2t2")])
    t = _mvit(t, tm.mvit[2], gelu_mode, attn_impl, "mv2")
    c3 = L.Add(name=_nm("fuse3"))([c3, _tr(t, tm.t2c[2], "t2c2")])

    fc = L.GlobalAveragePooling2D(name="gap_cnn")(_tr(c3, tm.cnn_last, "cnnlast"))
    ft = L.GlobalAveragePooling2D(name="gap_trf")(_tr(t, tm.t_head, "thead"))
    f = L.Concatenate(axis=-1, name="feat_cat")([fc, ft])
    # logits = 0.5 * (fc_c(fc) + fc_t(ft))  ==  one FC on [fc, ft]   (dropout is identity at inference)
    W = 0.5 * np.concatenate([_np(tm.fc_c.weight), _np(tm.fc_t.weight)], axis=1)
    b = 0.5 * (_np(tm.fc_c.bias) + _np(tm.fc_t.bias))
    out = _dense(f, W, b, "head")
    return tf.keras.Model(inp, out, name="TFCWDA_deploy")


def keras_parity(tm, km, arr_u8, n):
    # Gate against weight-copy / layout bugs: Keras (float) vs PyTorch (float) on the same raw pixels.
    x = arr_u8[:n]
    ref = torch_logits_from_raw(tm, x, "cpu")
    got = np.concatenate([km(x[i:i + 1].astype(np.float32) / 255.0, training=False).numpy() for i in range(len(x))], 0)
    mad = float(np.abs(ref - got).max())
    rel = mad / (float(np.abs(ref).max()) + 1e-6)
    agree = float((ref.argmax(1) == got.argmax(1)).mean())
    return dict(max_abs_diff=mad, rel_diff=rel, top1_agree=agree)

print("Keras translator ready.")

Keras translator ready.


In [9]:
# ============================================================
# PART 6C: TFLITE CONVERSION (FP32 + full INT8), INTERPRETER EVALUATION, AUDIT
# ============================================================
WORKING_VARIANT = None      # remembered after the first successful export, tried first for the next seeds


def _make_rep_gen(cal_u8):
    def gen():
        for i in range(cal_u8.shape[0]):
            yield [cal_u8[i:i + 1].astype(np.float32) / 255.0]       # model input is raw [0,1]
    return gen


def _convert(km, cal_u8=None):
    conv = tf.lite.TFLiteConverter.from_keras_model(km)
    if cal_u8 is not None:                                           # identical settings to im-main Part 10B
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = _make_rep_gen(cal_u8)
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8
        conv.inference_output_type = tf.int8
    return conv.convert()


def export_tflite_int8(tm_cpu, cal_u8, parity_arr, tag="", variants=None):
    global WORKING_VARIANT
    order = list(variants or EXPORT_VARIANTS)
    if WORKING_VARIANT in order:
        order.remove(WORKING_VARIANT)
        order.insert(0, WORKING_VARIANT)
    for gelu_mode, attn_impl in order:
        print(f"  [{tag}] variant gelu={gelu_mode} attention={attn_impl}")
        tf.keras.backend.clear_session(); gc.collect()
        t0 = time.time()
        km = build_keras_tfc(tm_cpu, IMG_SIZE, gelu_mode, attn_impl)      # translation bugs are NOT caught: they must be loud
        par = keras_parity(tm_cpu, km, parity_arr, PARITY_N)
        print(f"    Keras params {km.count_params():,} | parity vs PyTorch: max|dlogit|={par['max_abs_diff']:.2e} "
              f"(rel {par['rel_diff']:.2e}), top-1 agreement {par['top1_agree']*100:.1f}%")
        if par["rel_diff"] > PARITY_FAIL_REL:
            raise RuntimeError(f"Keras model does not reproduce the PyTorch model (rel diff {par['rel_diff']:.3f}); "
                               f"weight copy / layout bug - refusing to quantize.")
        try:
            fp32_bytes = _convert(km, None)
            int8_bytes = _convert(km, cal_u8)
        except Exception as e:
            print(f"    conversion failed for this variant: {type(e).__name__}: {str(e)[:300]}")
            continue
        WORKING_VARIANT = (gelu_mode, attn_impl)
        print(f"    OK in {time.time()-t0:.0f}s | FP32 tflite {len(fp32_bytes)/1024:.1f} KB | INT8 tflite {len(int8_bytes)/1024:.1f} KB")
        return dict(fp32_bytes=fp32_bytes, int8_bytes=int8_bytes, gelu=gelu_mode, attn=attn_impl, parity=par)
    print(f"  [{tag}] !! no variant converted to full INT8 !!")
    return None


def tflite_logits(model_bytes, arr_u8, int8, threads=TFLITE_THREADS):
    # Runs the REAL TFLite interpreter. For INT8 the input is quantized with the model's own scale/zero-point and
    # the int8 output is dequantized, exactly what a device would do.
    it = tf.lite.Interpreter(model_content=model_bytes, num_threads=threads)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    (s_i, z_i), (s_o, z_o) = ind["quantization"], outd["quantization"]
    if int8:
        assert ind["dtype"] == np.int8 and outd["dtype"] == np.int8, "model I/O is not int8"
    out = np.zeros((len(arr_u8), outd["shape"][-1]), np.float32)
    for i in range(len(arr_u8)):
        a = arr_u8[i:i + 1].astype(np.float32) / 255.0
        if int8:
            a = np.clip(np.round(a / s_i + z_i), -128, 127).astype(np.int8)
        it.set_tensor(ind["index"], a)
        it.invoke()
        o = it.get_tensor(outd["index"])[0]
        out[i] = (o.astype(np.float32) - z_o) * s_o if int8 else o
    return out


def audit_tflite(model_bytes, label):
    it = tf.lite.Interpreter(model_content=model_bytes)
    it.allocate_tensors()
    ind, outd = it.get_input_details()[0], it.get_output_details()[0]
    cnt, nbytes, big_float = Counter(), Counter(), []
    for d in it.get_tensor_details():
        dt = np.dtype(d["dtype"])
        n = int(np.prod(d["shape"])) if len(d["shape"]) else 1
        cnt[dt.name] += 1
        nbytes[dt.name] += n * dt.itemsize
        if dt == np.float32 and n >= 1000:
            big_float.append((d["name"][:60], n))
    print(f"\n--- INT8 audit: {label} ---")
    print(f"file size           : {len(model_bytes)/1024:.1f} KB")
    print(f"input               : dtype={np.dtype(ind['dtype']).name} shape={list(ind['shape'])} (scale, zero_point)={ind['quantization']}")
    print(f"output              : dtype={np.dtype(outd['dtype']).name} shape={list(outd['shape'])} (scale, zero_point)={outd['quantization']}")
    print(f"tensors by dtype    : {dict(cnt)}")
    print(f"MB by dtype         : { {k: round(v/1024**2, 2) for k, v in nbytes.items()} }  (weights + activation buffers)")
    if big_float:
        print(f"WARNING float32 tensors with >=1000 elements: {big_float[:6]}")
    else:
        print("no float32 tensor with >=1000 elements (no un-quantized weights / activations)")
    try:
        ops = Counter(d["op_name"] for d in it._get_ops_details())
        print(f"ops ({sum(ops.values())} nodes)  : {dict(sorted(ops.items()))}")
        bad = [o for o in ops if "flex" in o.lower() or "custom" in o.lower()]
        print("Flex/custom ops     :", bad if bad else "none")
    except Exception as e:
        print("op list not available in this TF build:", e)


print("TFLite export / evaluation helpers ready.")

TFLite export / evaluation helpers ready.


## Part 6b - Smoke test of the whole export (before the long training)

Builds the Keras model from an **untrained** `TFCNet` (pretrained MobileNetV2 trunk, random MobileViT / heads), runs the
parity gate, converts to FP32 and full-INT8 TFLite with a small calibration set, audits the file and runs the interpreter.
If this cell raises, nothing is wasted: fix it here before Part 7 starts training. The accuracy printed here is meaningless
(the net is untrained); only "does it convert, is it int8, does it run" matters. The numbers `FP32-vs-INT8 top-1 agreement` and
`logit correlation` are a first look at how much PTQ perturbs this architecture.

In [10]:
# ============================================================
# PART 6b: EXPORT SMOKE TEST (untrained network)
# ============================================================
if RUN_INT8_EXPORT and EXPORT_SMOKE_TEST:
    set_seed(0)
    _tm = TFCNet(NUM_CLASSES).cpu().eval()
    _cal = make_calibration(32)
    _res = export_tflite_int8(_tm, _cal, RAW["tgt_val"], tag="smoke")
    if _res is None:
        raise RuntimeError("Smoke test: no export variant converted to full INT8. See the messages above; "
                           "do not start the training before this is fixed.")
    audit_tflite(_res["int8_bytes"], "smoke test (untrained net)")

    _x = RAW["tgt_test"][:24]
    _lf = tflite_logits(_res["fp32_bytes"], _x, False)
    _li = tflite_logits(_res["int8_bytes"], _x, True)
    _lt = torch_logits_from_raw(_tm, _x, "cpu")
    print(f"\nFP32 TFLite vs PyTorch : max|dlogit| = {np.abs(_lf - _lt).max():.2e}")
    print(f"INT8 TFLite vs FP32    : top-1 agreement = {(_lf.argmax(1) == _li.argmax(1)).mean()*100:.1f}% | "
          f"logit correlation = {np.corrcoef(_lf.ravel(), _li.ravel())[0, 1]:.4f}")
    print(f"Working export variant : gelu={_res['gelu']}  attention={_res['attn']}")
    print("\nSmoke test passed: the export pipeline works end to end.")
    del _tm, _cal, _res, _x, _lf, _li, _lt
    gc.collect()
else:
    print("Smoke test skipped.")

  [smoke] variant gelu=exact attention=bmm
    Keras params 4,619,179 | parity vs PyTorch: max|dlogit|=8.20e-07 (rel 1.86e-06), top-1 agreement 100.0%
Saved artifact at '/tmp/tmpnegknzb8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  139443421667472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421665552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421665936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669392:

W0000 00:00:1792239587.144683      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792239587.144729      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1792239587.465232      24 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


Saved artifact at '/tmp/tmpx6iqxnz7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  139443421667472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421665552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421665936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421668240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421667664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139443421669584: TensorSpec

W0000 00:00:1792239608.358015      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792239608.358051      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


    OK in 107s | FP32 tflite 14751.0 KB | INT8 tflite 4399.3 KB

--- INT8 audit: smoke test (untrained net) ---
file size           : 4399.3 KB
input               : dtype=int8 shape=[np.int32(1), np.int32(224), np.int32(224), np.int32(3)] (scale, zero_point)=(0.003921568859368563, -128)
output              : dtype=int8 shape=[np.int32(1), np.int32(3)] (scale, zero_point)=(0.0026681534945964813, -40)
tensors by dtype    : {'int8': 736, 'int32': 132, 'float32': 42}
MB by dtype         : {'int8': 27.79, 'int32': 0.09, 'float32': 0.04}  (weights + activation buffers)
no float32 tensor with >=1000 elements (no un-quantized weights / activations)
ops (713 nodes)  : {'ADD': 76, 'BATCH_MATMUL': 18, 'CONCATENATION': 7, 'CONV_2D': 56, 'DELEGATE': 70, 'DEPTHWISE_CONV_2D': 17, 'DEQUANTIZE': 21, 'FULLY_CONNECTED': 55, 'GELU': 9, 'LOGISTIC': 18, 'MEAN': 44, 'MUL': 82, 'NEG': 21, 'PACK': 9, 'PAD': 7, 'QUANTIZE': 21, 'RESHAPE': 69, 'RESIZE_BILINEAR': 2, 'RSQRT': 21, 'SOFTMAX': 9, 'SQUARED_DIFFERENCE'

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



FP32 TFLite vs PyTorch : max|dlogit| = 9.69e-07
INT8 TFLite vs FP32    : top-1 agreement = 79.2% | logit correlation = 0.9567
Working export variant : gelu=exact  attention=bmm

Smoke test passed: the export pipeline works end to end.


## Part 7 - Multi-seed run: train, evaluate FP32, export INT8, evaluate INT8

For every seed: train (Part 5) -> PyTorch FP32 evaluation on the held-out target test set -> rebuild in Keras -> parity gate ->
FP32 and full-INT8 TFLite -> evaluate the **real INT8 interpreter** on the same 79 target-test images (and, optionally, the
564 source-test images). The target test set is touched once per model. Per-image predictions of the PyTorch model, the FP32 TFLite
and the INT8 TFLite are saved for every seed, so a paired test (e.g. McNemar) against your other models, and between FP32 and INT8, is possible.
The `.tflite` files are written to `models/`.

In [11]:
# ============================================================
# PART 7: MULTI-SEED RUN + FP32 EVALUATION + INT8 EXPORT + INT8 EVALUATION
# ============================================================
_NAN = float("nan")
_AUDIT = {"done": False}
INT8_EMPTY = dict(int8_ok=False, int8_gelu="", int8_attn="", keras_parity_maxdiff=_NAN, keras_parity_agree=_NAN,
                  tflite_fp32_kb=_NAN, int8_kb=_NAN, tflite_fp32_tgt_acc=_NAN,
                  int8_tgt_acc=_NAN, int8_tgt_f1_macro=_NAN, int8_tgt_f1_weighted=_NAN, int8_tgt_kappa=_NAN,
                  int8_tgt_bal_acc_paper=_NAN, int8_ci_lo=_NAN, int8_ci_hi=_NAN,
                  int8_src_test_acc=_NAN, int8_agree_with_fp32=_NAN, int8_acc_drop=_NAN)


def int8_stage(model, seed, tl, tp_torch):
    # Returns (row_fields, per-image prediction columns).
    tm = copy.deepcopy(model).cpu().eval()
    cal = make_calibration(CAL_N_SRC)
    res = export_tflite_int8(tm, cal, RAW["tgt_val"], tag=f"seed{seed}")
    if res is None:
        return dict(INT8_EMPTY), {}

    p_fp32 = f"{MODELS_DIR}/tfcwda_seed{seed}_fp32.tflite"
    p_int8 = f"{MODELS_DIR}/tfcwda_seed{seed}_int8.tflite"
    with open(p_fp32, "wb") as f: f.write(res["fp32_bytes"])
    with open(p_int8, "wb") as f: f.write(res["int8_bytes"])

    assert np.array_equal(RAW["tgt_test_labels"], tl), "raw target-test order differs from the loader order"
    lg_f = tflite_logits(res["fp32_bytes"], RAW["tgt_test"], False)
    lg_i = tflite_logits(res["int8_bytes"], RAW["tgt_test"], True)
    pf, pi = lg_f.argmax(1), lg_i.argmax(1)
    met = full_metrics(tl, pi)
    _, lo, hi = bootstrap_ci(tl, pi)

    src_acc = _NAN
    if INT8_EVAL_SRC:
        lg_s = tflite_logits(res["int8_bytes"], RAW["src_test"], True)
        src_acc = accuracy_score(RAW["src_test_labels"], lg_s.argmax(1))

    out = dict(int8_ok=True, int8_gelu=res["gelu"], int8_attn=res["attn"],
               keras_parity_maxdiff=res["parity"]["max_abs_diff"], keras_parity_agree=res["parity"]["top1_agree"],
               tflite_fp32_kb=len(res["fp32_bytes"]) / 1024, int8_kb=len(res["int8_bytes"]) / 1024,
               tflite_fp32_tgt_acc=accuracy_score(tl, pf),
               int8_tgt_acc=met["acc"], int8_tgt_f1_macro=met["f1_macro"], int8_tgt_f1_weighted=met["f1_weighted"],
               int8_tgt_kappa=met["kappa"], int8_tgt_bal_acc_paper=met["bal_acc_paper"],
               int8_ci_lo=lo, int8_ci_hi=hi, int8_src_test_acc=src_acc,
               int8_agree_with_fp32=float((pi == tp_torch).mean()),
               int8_acc_drop=accuracy_score(tl, tp_torch) - met["acc"])
    if not _AUDIT["done"]:
        audit_tflite(res["int8_bytes"], f"seed {seed}, {p_int8}")
        _AUDIT["done"] = True
    del res, tm
    gc.collect()
    return out, dict(pred_tflite_fp32=pf, pred_int8=pi)


rows = []
for s in SEEDS:
    print(f"\n{'#'*100}\n# SEED {s}  (TFC-WDA, Tunio et al. reproduction)\n{'#'*100}")
    m, hist, best_sm, secs = train_one_run(s, tag=f"tfcwda_seed{s}")

    _, src_test_acc, src_test_f1, _, _   = evaluate(m, src_test_loader, cls_criterion, DEVICE)
    _, tgt_test_acc, tgt_test_f1, tp, tl = evaluate(m, tgt_test_loader, cls_criterion, DEVICE)
    bmean, blo, bhi = bootstrap_ci(tl, tp)
    met = full_metrics(tl, tp)

    row = dict(seed=s, src_test_acc=src_test_acc, src_test_f1=src_test_f1,
               tgt_test_acc=tgt_test_acc, tgt_test_f1=tgt_test_f1,
               boot_mean=bmean, ci_lo=blo, ci_hi=bhi, train_seconds=secs,
               **{f"tgt_{k}": v for k, v in met.items() if k not in ("acc",)})
    print(f"  seed {s}: FP32  SRC-TEST={src_test_acc:.4f}  TGT-TEST={tgt_test_acc:.4f} "
          f"(95% CI [{blo:.4f}, {bhi:.4f}])  macroF1={met['f1_macro']:.4f}  kappa={met['kappa']:.4f}")

    pred_cols = {'path': tgt_test_idx.img_paths, 'label': tl, 'pred_fp32_torch': tp}
    torch.save(m.state_dict(), f"{MODELS_DIR}/tfcwda_seed{s}.pth")

    if RUN_INT8_EXPORT:
        print(f"\n  --- INT8 export + evaluation, seed {s} ---")
        try:
            i8, extra = int8_stage(m, s, tl, tp)
        except Exception as e:                       # never lose a finished training run because of the export
            print(f"  !! INT8 stage crashed for seed {s}: {type(e).__name__}: {e}")
            traceback.print_exc()
            print(f"  The trained FP32 weights are saved: {MODELS_DIR}/tfcwda_seed{s}.pth (re-run Parts 6A-6C and export from them).")
            i8, extra = dict(INT8_EMPTY), {}
        row.update(i8)
        pred_cols.update(extra)
        if i8["int8_ok"]:
            print(f"  seed {s}: INT8  TGT-TEST={i8['int8_tgt_acc']:.4f} (95% CI [{i8['int8_ci_lo']:.4f}, {i8['int8_ci_hi']:.4f}])  "
                  f"macroF1={i8['int8_tgt_f1_macro']:.4f} | drop vs FP32 = {i8['int8_acc_drop']*100:+.2f} pts | "
                  f"agree with FP32 = {i8['int8_agree_with_fp32']*100:.1f}% | file = {i8['int8_kb']:.1f} KB "
                  f"(FP32 tflite {i8['tflite_fp32_kb']:.1f} KB) | variant gelu={i8['int8_gelu']}/attn={i8['int8_attn']}")
        else:
            print(f"  seed {s}: INT8 export FAILED - no INT8 number exists for this seed.")

    rows.append(row)
    pd.DataFrame(rows).to_csv(f"{TABLES_DIR}/tfcwda_runs_partial.csv", index=False)   # progress is never lost
    pd.DataFrame(pred_cols).to_csv(f"{TABLES_DIR}/tfcwda_tgt_test_predictions_seed{s}.csv", index=False)

    del m
    torch.cuda.empty_cache(); gc.collect()

tfc_df = pd.DataFrame(rows)
tfc_df.to_csv(f"{TABLES_DIR}/tfcwda_runs.csv", index=False)
print("\n" + tfc_df[['seed', 'src_test_acc', 'tgt_test_acc', 'tgt_f1_macro', 'tgt_kappa',
                     'ci_lo', 'ci_hi', 'train_seconds']].to_string(index=False))

print(f"\n{'='*70}\nTFC-WDA FP32 (PyTorch) -- cross-seed summary (mean +/- std over {len(SEEDS)} seeds: {SEEDS})\n{'='*70}")
for col, nm in [('src_test_acc', 'Source test acc'), ('tgt_test_acc', 'Target test acc'),
                ('tgt_f1_macro', 'Target macro F1'), ('tgt_bal_acc_paper', 'Target bal. acc (paper def.)'),
                ('tgt_kappa', "Target Cohen's kappa")]:
    sd = tfc_df[col].std() if len(SEEDS) > 1 else float('nan')
    print(f"{nm:30s}: {tfc_df[col].mean()*100:.2f} +/- {sd*100:.2f} %")

if RUN_INT8_EXPORT:
    ok = tfc_df["int8_ok"].astype(bool)
    print(f"\n{'='*70}\nTFC-WDA INT8 (real TFLite, full-integer) -- {int(ok.sum())}/{len(tfc_df)} seeds exported\n{'='*70}")
    if ok.any():
        d8 = tfc_df[ok]
        for col, nm in [('int8_tgt_acc', 'INT8 target test acc'), ('int8_tgt_f1_macro', 'INT8 target macro F1'),
                        ('int8_tgt_kappa', "INT8 target Cohen's kappa"), ('int8_src_test_acc', 'INT8 source test acc'),
                        ('int8_acc_drop', 'Accuracy drop FP32 -> INT8 (points)'),
                        ('int8_agree_with_fp32', 'INT8-vs-FP32 prediction agreement')]:
            sd = d8[col].std() if len(d8) > 1 else float('nan')
            print(f"{nm:38s}: {d8[col].mean()*100:.2f} +/- {sd*100:.2f} %")
        print(f"{'INT8 .tflite size (measured)':38s}: {d8.int8_kb.mean():.1f} KB (min {d8.int8_kb.min():.1f}, max {d8.int8_kb.max():.1f})")
        print(f"{'FP32 .tflite size (measured)':38s}: {d8.tflite_fp32_kb.mean():.1f} KB")
        print(f"{'export variants used':38s}: {sorted(set(zip(d8.int8_gelu, d8.int8_attn)))}")


####################################################################################################
# SEED 42  (TFC-WDA, Tunio et al. reproduction)
####################################################################################################
  [tfcwda_seed42] seed=42 | epochs<=50 | selection=smoothed_target_val | DA=ON
Ep   1/50 | ce=0.343 W=1.8824 upl=-0.602 lam=0.010 | train_acc=0.945 | SRC-VAL=0.989 | TGT-VAL=0.519 (sm=0.519) f1=0.494 | 60s
Ep   5/50 | ce=0.079 W=0.8278 upl=-0.672 lam=0.046 | train_acc=0.992 | SRC-VAL=0.977 | TGT-VAL=0.608 (sm=0.565) f1=0.579 | 290s
Ep  10/50 | ce=0.088 W=0.8468 upl=-0.673 lam=0.076 | train_acc=0.992 | SRC-VAL=0.993 | TGT-VAL=0.582 (sm=0.532) f1=0.571 | 574s
Ep  15/50 | ce=0.057 W=0.5903 upl=-0.677 lam=0.090 | train_acc=0.996 | SRC-VAL=0.970 | TGT-VAL=0.570 (sm=0.603) f1=0.574 | 859s
Ep  20/50 | ce=0.087 W=0.6840 upl=-0.675 lam=0.096 | train_acc=0.991 | SRC-VAL=0.949 | TGT-VAL=0.658 (sm=0.603) f1=0.640 | 1141s
Ep  25/50 | ce=0.045 W=0.3653 

W0000 00:00:1792241867.462788      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792241867.462843      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Saved artifact at '/tmp/tmplnr3b4v6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  139442294938640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294939792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294939984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294936528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294933648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294940560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294941904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294941520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294939408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294933072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442294940176: TensorSpec

W0000 00:00:1792241888.515125      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792241888.515156      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


    OK in 250s | FP32 tflite 17950.4 KB | INT8 tflite 5343.6 KB

--- INT8 audit: seed 42, /kaggle/working/tfcwda_baseline/models/tfcwda_seed42_int8.tflite ---
file size           : 5343.6 KB
input               : dtype=int8 shape=[np.int32(1), np.int32(224), np.int32(224), np.int32(3)] (scale, zero_point)=(0.003921568859368563, -128)
output              : dtype=int8 shape=[np.int32(1), np.int32(3)] (scale, zero_point)=(0.06130354478955269, -34)
tensors by dtype    : {'int8': 832, 'int32': 168, 'float32': 42}
MB by dtype         : {'int8': 29.16, 'int32': 0.11, 'float32': 0.04}  (weights + activation buffers)
no float32 tensor with >=1000 elements (no un-quantized weights / activations)
ops (734 nodes)  : {'ADD': 97, 'BATCH_MATMUL': 18, 'CONCATENATION': 7, 'CONV_2D': 56, 'DELEGATE': 70, 'DEPTHWISE_CONV_2D': 17, 'DEQUANTIZE': 21, 'FULLY_CONNECTED': 55, 'GELU': 9, 'LOGISTIC': 18, 'MEAN': 44, 'MUL': 82, 'NEG': 21, 'PACK': 9, 'PAD': 7, 'QUANTIZE': 21, 'RESHAPE': 69, 'RESIZE_BILINEAR': 2, 'R

W0000 00:00:1792243139.259013      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792243139.259072      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


... (3 outputs left out)

W0000 00:00:1792245267.228985      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792245267.229047      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Saved artifact at '/tmp/tmpq3iws307'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  139442289987024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289999696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289992208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289988944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289985104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289985488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289993168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289986256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289984528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289987984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139442289992784: TensorSpec

W0000 00:00:1792245288.281968      24 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1792245288.281999      24 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


    OK in 250s | FP32 tflite 17950.4 KB | INT8 tflite 5343.0 KB
  seed 2024: INT8  TGT-TEST=0.5949 (95% CI [0.4937, 0.7089])  macroF1=0.5816 | drop vs FP32 = -2.53 pts | agree with FP32 = 92.4% | file = 5343.0 KB (FP32 tflite 17950.4 KB) | variant gelu=exact/attn=bmm

 seed  src_test_acc  tgt_test_acc  tgt_f1_macro  tgt_kappa    ci_lo    ci_hi  train_seconds
   42      0.978723      0.658228      0.641811   0.491052 0.556962 0.759494    2166.172777
 1337      0.991135      0.531646      0.452722   0.298199 0.417722 0.645570     982.598646
 2024      0.978723      0.569620      0.569755   0.355102 0.468354 0.683544    1841.001589

TFC-WDA FP32 (PyTorch) -- cross-seed summary (mean +/- std over 3 seeds: [42, 1337, 2024])
Source test acc               : 98.29 +/- 0.72 %
Target test acc               : 58.65 +/- 6.50 %
Target macro F1               : 55.48 +/- 9.54 %
Target bal. acc (paper def.)  : 68.88 +/- 5.14 %
Target Cohen's kappa          : 38.15 +/- 9.91 %

TFC-WDA INT8 (real TFLite

## Part 8 - Optional ablation: no Wasserstein / no UPL

Set `RUN_ABLATION = True` in Part 0 to run it. It keeps the fused architecture, the dual classifiers, the sampler and
the elastic net, and removes the Wasserstein critic and UPL. This shows how much of the number comes from the adaptation
and how much from the backbone. The ablation is evaluated in FP32 only (it is not part of the deployment comparison).

In [12]:
# ============================================================
# PART 8: OPTIONAL ABLATION (fusion only, no domain adaptation), FP32 only
# ============================================================
abl_df = None
if RUN_ABLATION:
    abl_rows = []
    for s in SEEDS:
        print(f"\n{'#'*100}\n# SEED {s}  (fusion network, DA OFF)\n{'#'*100}")
        m, _, _, secs = train_one_run(s, use_da=False, tag=f"tfc_noDA_seed{s}")
        _, sa, _, _, _       = evaluate(m, src_test_loader, cls_criterion, DEVICE)
        _, ta, tf1, tp, tl   = evaluate(m, tgt_test_loader, cls_criterion, DEVICE)
        abl_rows.append(dict(seed=s, src_test_acc=sa, tgt_test_acc=ta, tgt_test_f1=tf1, train_seconds=secs))
        print(f"  seed {s}: SRC-TEST={sa:.4f}  TGT-TEST={ta:.4f}")
        del m
        torch.cuda.empty_cache()
    abl_df = pd.DataFrame(abl_rows)
    abl_df.to_csv(f"{TABLES_DIR}/tfc_noDA_runs.csv", index=False)
    table = pd.DataFrame([
        dict(Variant="Fusion network, no Wasserstein / UPL", Tgt_Acc_Mean=abl_df.tgt_test_acc.mean()*100,
             Tgt_Acc_Std=abl_df.tgt_test_acc.std()*100),
        dict(Variant="Fusion network + Wasserstein + UPL (full)", Tgt_Acc_Mean=tfc_df.tgt_test_acc.mean()*100,
             Tgt_Acc_Std=tfc_df.tgt_test_acc.std()*100),
    ])
    print("\nAblation (target-test accuracy, mean +/- std):")
    print(table.to_string(index=False))
    table.to_csv(f"{TABLES_DIR}/ablation_da_vs_no_da.csv", index=False)
else:
    print("RUN_ABLATION=False: skipped.")

RUN_ABLATION=False: skipped.


## Part 9 - Size and training-time memory

Sizes are now **measured files**: the FP32 number is the parameter count x 4 bytes (PyTorch checkpoint size) and the INT8 number is the
size of the `.tflite` file that was actually evaluated in Part 7. Only the classifier is counted, the critic is training-only.
The memory probe runs a short training (critic included) and reads PyTorch's peak GPU memory counter.

In [13]:
# ============================================================
# PART 9: SIZE + TRAINING MEMORY
# ============================================================
_m = TFCNet(NUM_CLASSES)
n_params = count_params(_m)
n_critic_params = count_params(DomainCritic(_m.feat_dim))
size_fp32_mb = n_params * 4 / 1024 ** 2
del _m
print(f"Classifier params : {n_params:,}   (+ {n_critic_params:,} critic params, training only)")
print(f"FP32 size         : {size_fp32_mb:.2f} MB  (params x 4 bytes)")

int8_kb_mean = None
if RUN_INT8_EXPORT and tfc_df["int8_ok"].astype(bool).any():
    _d = tfc_df[tfc_df["int8_ok"].astype(bool)]
    int8_kb_mean = float(_d.int8_kb.mean())
    print(f"INT8 .tflite size : {int8_kb_mean:.1f} KB = {int8_kb_mean/1024:.2f} MB   (MEASURED file, mean of {len(_d)} seed(s))")
    for _, r_ in _d.iterrows():
        print(f"   seed {int(r_.seed)}: {r_.int8_kb:.1f} KB  (FP32 tflite {r_.tflite_fp32_kb:.1f} KB)")
    print(f"Compression FP32 tflite -> INT8 tflite: {_d.tflite_fp32_kb.mean()/int8_kb_mean:.2f}x")
else:
    print("INT8 .tflite size : NOT MEASURED (INT8 export disabled or failed) - no estimate is reported.")

peak_mem_mb, probe_seconds = None, None
if DEVICE == "cuda":
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _t0 = time.time()
    _pm, _, _, _ = train_one_run(SEEDS[0], epochs=PROBE_EPOCHS, verbose=False, tag="memory_probe")
    torch.cuda.synchronize()
    probe_seconds = time.time() - _t0
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    del _pm
    torch.cuda.empty_cache()
    print(f"Peak GPU memory   : {peak_mem_mb:.1f} MB  ({PROBE_EPOCHS}-epoch probe, {probe_seconds:.0f}s)")
else:
    print("No GPU: memory probe skipped.")

Classifier params : 4,580,190   (+ 459,521 critic params, training only)
FP32 size         : 17.47 MB  (params x 4 bytes)
INT8 .tflite size : 5343.0 KB = 5.22 MB   (MEASURED file, mean of 3 seed(s))
   seed 42: 5343.6 KB  (FP32 tflite 17950.4 KB)
   seed 1337: 5342.4 KB  (FP32 tflite 17950.4 KB)
   seed 2024: 5343.0 KB  (FP32 tflite 17950.4 KB)
Compression FP32 tflite -> INT8 tflite: 3.36x
Peak GPU memory   : 6214.9 MB  (2-epoch probe, 117s)


## Part 10 - Result for the comparison table

The im-main numbers below are copied from the multi-seed summary and the INT8 export printed by your im-main notebook
(target accuracy 75.53 +/- 1.58 over 3 seeds, macro F1 74.41, source accuracy 99.41, 167,131 parameters, INT8 TFLite 245.6 KB, and 75.95 % target accuracy
for the INT8 TFLite of the single exported model).
**They are hard-coded. Update them if you re-run im-main.**

In [14]:
# ============================================================
# PART 10: RESULT + LATEX ROW + COMPARISON WITH im-main
# ============================================================
acc_mean, acc_std = tfc_df.tgt_test_acc.mean(), tfc_df.tgt_test_acc.std()
_ok = tfc_df["int8_ok"].astype(bool) if "int8_ok" in tfc_df else pd.Series([False] * len(tfc_df))
d8 = tfc_df[_ok] if _ok.any() else None

result = dict(
    method="TFC-WDA (Tunio et al., 2024), this reproduction",
    seeds=SEEDS,
    params=n_params,
    size_fp32_mb=round(size_fp32_mb, 2),
    int8_tflite_kb_measured=None if d8 is None else round(float(d8.int8_kb.mean()), 1),
    int8_seeds_exported=0 if d8 is None else int(len(d8)),
    src_test_acc=round(float(tfc_df.src_test_acc.mean()), 4),
    tgt_test_acc_fp32_mean=round(float(acc_mean), 4),
    tgt_test_acc_fp32_std=round(float(acc_std), 4),
    tgt_f1_macro_fp32=round(float(tfc_df.tgt_f1_macro.mean()), 4),
    tgt_test_acc_int8_mean=None if d8 is None else round(float(d8.int8_tgt_acc.mean()), 4),
    tgt_test_acc_int8_std=None if d8 is None else round(float(d8.int8_tgt_acc.std()), 4),
    tgt_f1_macro_int8=None if d8 is None else round(float(d8.int8_tgt_f1_macro.mean()), 4),
    train_peak_mem_mb=None if peak_mem_mb is None else round(peak_mem_mb, 1),
)
for k, v in result.items():
    print(f"{k:26s}: {v}")
pd.DataFrame([result]).to_csv(f"{TABLES_DIR}/tfcwda_result.csv", index=False)

# ---- LaTeX row: INT8 column = MEASURED .tflite size, accuracy = real INT8 interpreter on the target test set ----
if d8 is not None:
    acc8_txt = f"{d8.int8_tgt_acc.mean()*100:.2f} $\\pm$ {d8.int8_tgt_acc.std()*100:.2f}"
    size_txt = f"{d8.int8_kb.mean()/1024:.2f} MB"
    latex_row = (r"TFC-WDA~\cite{tunio2024tfc} & MobileNetV2 + MobileViT fusion & "
                 r"Wasserstein critic + UPL & " + acc8_txt + f" & {size_txt} \\\\")
    print("\nLaTeX row (INT8 accuracy and INT8 size are both MEASURED on the .tflite file):\n")
    print(latex_row)
else:
    print("\nNo INT8 model was exported, so no INT8 LaTeX row is produced (no estimate is substituted).")

# ---- im-main numbers: UPDATE if you re-run im-main ----
IM_PARAMS          = 167_131
IM_SIZE_INT8_KB    = 245.6        # measured full-INT8 TFLite
IM_SRC_TEST_ACC    = 0.9941
IM_TGT_MEAN        = 0.7553       # FP32 PyTorch, 3 seeds
IM_TGT_STD         = 0.0158
IM_TGT_F1_MACRO    = 0.7441
IM_INT8_TGT_SINGLE = 0.7595       # INT8 TFLite of the single exported model (not a 3-seed number)

cmp_rows = [
    dict(Method="TFC-WDA (Tunio et al. 2024, this reproduction)",
         Params=n_params, Size_FP32_MB=round(size_fp32_mb, 2),
         Size_INT8_TFLite_KB=("not measured" if d8 is None else f"{d8.int8_kb.mean():.1f}"),
         Src_Test_Acc=f"{tfc_df.src_test_acc.mean()*100:.2f}%",
         Tgt_FP32_Acc=f"{acc_mean*100:.2f} +/- {acc_std*100:.2f}%",
         Tgt_INT8_Acc=("n/a" if d8 is None else f"{d8.int8_tgt_acc.mean()*100:.2f} +/- {d8.int8_tgt_acc.std()*100:.2f}%"),
         Tgt_F1_macro_FP32=f"{tfc_df.tgt_f1_macro.mean()*100:.2f}%"),
    dict(Method="MicroBrainNet-DA (im-main)",
         Params=IM_PARAMS, Size_FP32_MB=round(IM_PARAMS*4/1024**2, 2),
         Size_INT8_TFLite_KB=f"{IM_SIZE_INT8_KB:.1f}",
         Src_Test_Acc=f"{IM_SRC_TEST_ACC*100:.2f}%",
         Tgt_FP32_Acc=f"{IM_TGT_MEAN*100:.2f} +/- {IM_TGT_STD*100:.2f}%",
         Tgt_INT8_Acc=f"{IM_INT8_TGT_SINGLE*100:.2f}% (1 model)",
         Tgt_F1_macro_FP32=f"{IM_TGT_F1_MACRO*100:.2f}%"),
]
cmp_table = pd.DataFrame(cmp_rows)
print("\n" + "=" * 130)
print("  IDENTICAL-DATA COMPARISON: PlantVillage -> PlantDoc, 3-class split, 3 seeds (42, 1337, 2024)")
print("=" * 130)
print(cmp_table.to_string(index=False))
print(f"\nParameter ratio (TFC-WDA / im-main): {n_params / IM_PARAMS:.1f}x")
if d8 is not None:
    print(f"INT8 file size ratio (TFC-WDA / im-main): {d8.int8_kb.mean() / IM_SIZE_INT8_KB:.1f}x   (both are measured .tflite files)")
cmp_table.to_csv(f"{TABLES_DIR}/final_comparison_table.csv", index=False)

print("\nGenerated files:")
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, OUTPUT_DIR):<52s} {os.path.getsize(p)/1024:10.1f} KB")

method                    : TFC-WDA (Tunio et al., 2024), this reproduction
seeds                     : [42, 1337, 2024]
params                    : 4580190
size_fp32_mb              : 17.47
int8_tflite_kb_measured   : 5343.0
int8_seeds_exported       : 3
src_test_acc              : 0.9829
tgt_test_acc_fp32_mean    : 0.5865
tgt_test_acc_fp32_std     : 0.065
tgt_f1_macro_fp32         : 0.5548
tgt_test_acc_int8_mean    : 0.5992
tgt_test_acc_int8_std     : 0.0697
tgt_f1_macro_int8         : 0.563
train_peak_mem_mb         : 6214.9

LaTeX row (INT8 accuracy and INT8 size are both MEASURED on the .tflite file):

TFC-WDA~\cite{tunio2024tfc} & MobileNetV2 + MobileViT fusion & Wasserstein critic + UPL & 59.92 $\pm$ 6.97 & 5.22 MB \\

  IDENTICAL-DATA COMPARISON: PlantVillage -> PlantDoc, 3-class split, 3 seeds (42, 1337, 2024)
                                        Method  Params  Size_FP32_MB Size_INT8_TFLite_KB Src_Test_Acc    Tgt_FP32_Acc     Tgt_INT8_Acc Tgt_F1_macro_FP32
TFC-WDA (Tunio e

## Caveats to state in the manuscript

1. **Reconstruction, not original code.** The paper gives no code link. Unspecified details were assumed: critic design,
   critic steps, gradient-penalty weight, lambda, alpha, elastic-net strengths, MobileViT widths, rotation angle, filter sizes.
   They are in the config cell. No tuning was done on this split.
2. **Stem.** Sec. 3.2.2 describes a 7x7/stride-2 conv + max-pool stem. Table 2 and Table 4 say the backbone is MobileNetV2.
   This reproduction uses the pretrained MobileNetV2 as the CNN branch, so its own stem is used.
3. **"NFRegNet".** The paper presents L1/L2/elastic-net weight regularization as a normalizer-free alternative to batch norm.
   A weight penalty does not replace normalization, and a pretrained MobileNetV2 depends on its BN layers.
   Here the elastic-net penalty (Eqs. 5-7) is applied and BN is kept. Say this explicitly if you describe the baseline.
4. **UPL sign.** Eq. 14 as printed is the entropy of the wrong-class distribution, and Eq. 15 puts it inside a max over the critic.
   Read literally, minimizing Eq. 14 would sharpen the wrong-class distribution, which contradicts the paper's own example.
   The implementation follows the cited Guided Complement Entropy: the wrong-class distribution is flattened.
5. **Eq. 15 sign.** The extractor minimizes the critic's Wasserstein estimate (Eq. 12). The `-lambda * Max[...]` form of Eq. 15 is read that way.
6. **CDAN** is mentioned in the Introduction but does not appear in the objective, so it is not implemented.
7. **Target labels in training: none.** Only target-train images are used, without labels. Target-val labels are used for checkpoint selection only.
   Set `SELECTION = "last"` for a strict UDA number and say which one you report.
8. **Not comparable to the paper's 70.0%.** That number is a 27-class PlantDoc test on 452 images, with AC-PD crops of PlantDoc
   in training. This notebook uses your 3-class split. The paper's Fig. 9 also marks a "selected test accuracy" epoch,
   which suggests its checkpoint was chosen on test curves, whereas here selection uses target-val only.
9. **INT8 is now measured, and its method is the same as im-main's.** Full-integer post-training quantization through the native TF/Keras rebuild and
   `tf.lite.TFLiteConverter` (per-channel int8 conv weights, per-tensor int8 activations, int8 input/output, `TFLITE_BUILTINS_INT8`, no ONNX).
   It is **PTQ, not QAT**; LayerNorm, softmax and attention are known to be the most quantization-sensitive parts of a transformer, so a drop
   relative to FP32 is a genuine property of this architecture under the same PTQ recipe, and the FP32 -> INT8 drop is reported per seed.
10. **Calibration data.** 400 source-val + all 79 target-val images (labels not used); im-main used the same recipe with 511 images in total. No training or test image is used for calibration.
    The target images pass through the same median + Gaussian filter as in evaluation.
11. **Export variant.** The GELU (exact erf vs tanh-approximate) and attention (BATCH_MATMUL vs broadcast-mul/reduce-sum) form that converted to full INT8
    is stored per seed (`int8_gelu`, `int8_attn` in `tfcwda_runs.csv`). If a tanh-approximate GELU was needed, the FP32 TFLite differs from PyTorch by ~1e-3 in
    activations; the parity gate output shows the exact deviation. State the variant if you describe the deployment model.
12. **Size.** FP32 size is parameters x 4 bytes. INT8 size is the measured `.tflite` file. The critic is excluded (training only). The model is a server/mobile-class
    network: its INT8 file is far larger than im-main's, so the comparison is about accuracy per byte, not about both fitting the same MCU budget.
13. **Epochs.** 50, as in Table 2, with early stopping on the smoothed target-val rule.
14. **Small target test set (79 images).** Report mean +/- std over seeds and the bootstrap CI. One image is 1.27 accuracy points, so an INT8 drop of one or two
    images is within noise; use the saved per-image predictions (FP32 vs INT8) for a paired comparison.